# Análise do Manchester City 2020/21

## Preparação de Ambiente

In [113]:
import pandas as pd
import numpy as np
import warnings
import plotly.express as px
import plotly.graph_objects as go

pd.options.mode.chained_assignment = None
warnings.filterwarnings("ignore")
np.random.seed(4)

In [114]:
data = pd.read_csv("data.csv")
data["Min"] = data["Min"].apply(lambda x: x.replace(",", "")).astype(int)
data = data.loc[data["Min"] > 500].reset_index(drop=True)

# Separate metadata from stats before numeric conversion
meta = data.iloc[:, :11]
stats = data.iloc[:, 11:].apply(pd.to_numeric, errors="coerce")
data = pd.concat([meta, stats], axis=1).fillna(0)

## Dataset

Dados das 5 principais ligas europeias (2020/21) via FBRef. Filtro: > 500 minutos jogados.

Referência de métricas: https://x.com/ronanmann/status/1408504415690969089

In [115]:
data.head()

,Player,Nation,Pos,Squad,Comp,Age,Born,MP,Starts,Min,...,PrgDistCarry/90,ProgCarry/90,CarryIntoThird/90,CarryIntoBox/90,Miscontrol/90,Dispossessed/90,PassTarget/90,PassesReceived/90,PassRec%,ProgPassReceived/90
0,Patrick van Aanholt,nl NED,DF,Crystal Palace,eng Premier League,29.0,1990.0,22,20,1777,...,96.8,5.79,1.52,0.46,0.66,0.81,42.3,39.3,92.8,2.34
1,Yunis Abdelhamid,ma MAR,DF,Reims,fr Ligue 1,32.0,1987.0,33,33,2889,...,130.5,1.81,0.25,0.03,0.25,0.47,43.8,42.5,97.0,0.22
2,Pape Abou Cisse,sn SEN,DF,Saint-Étienne,fr Ligue 1,24.0,1995.0,14,14,1260,...,117.9,1.14,0.07,0.00,0.14,0.07,33.7,32.9,97.7,0.00
3,David Abraham,ar ARG,DF,Eint Frankfurt,de Bundesliga,34.0,1986.0,14,14,1222,...,92.7,3.24,0.66,0.00,0.22,0.00,34.8,33.4,96.0,0.59
4,Francesco Acerbi,it ITA,DF,Lazio,it Serie A,32.0,1988.0,32,32,2813,...,166.3,3.71,1.02,0.03,0.45,0.26,50.9,49.5,97.3,0.70


In [116]:
with open("metrics.txt", "w") as f:
    for col in data.columns:
        f.write(f"{col}\n")

In [117]:
data['Pos'].unique()

<StringArray>
['DF', 'DF,FW', 'DF,MF', 'FW', 'FW,DF', 'FW,MF', 'GK', 'MF', 'MF,DF', 'MF,FW']
Length: 10, dtype: str

In [118]:
city_players = data[data['Squad'] == 'Manchester City'].copy()
city_players

,Player,Nation,Pos,Squad,Comp,Age,Born,MP,Starts,Min,...,PrgDistCarry/90,ProgCarry/90,CarryIntoThird/90,CarryIntoBox/90,Miscontrol/90,Dispossessed/90,PassTarget/90,PassesReceived/90,PassRec%,ProgPassReceived/90
15,Nathan Aké,nl NED,DF,Manchester City,eng Premier League,25.0,1995.0,10,9,797,...,207.5,7.87,3.03,0.00,0.34,0.22,75.4,73.7,97.8,0.34
113,João Cancelo,pt POR,DF,Manchester City,eng Premier League,26.0,1994.0,28,27,2299,...,166.3,8.55,2.86,0.86,0.98,1.22,69.5,66.3,95.4,3.57
189,Rúben Dias,pt POR,DF,Manchester City,eng Premier League,23.0,1997.0,32,32,2843,...,255.1,7.28,0.54,0.00,0.22,0.16,76.2,74.9,98.3,0.35
366,Aymeric Laporte,fr FRA,DF,Manchester City,eng Premier League,26.0,1994.0,16,14,1344,...,274.1,9.40,1.41,0.00,0.20,0.13,74.4,73.4,98.6,0.13
445,Benjamin Mendy,fr FRA,DF,Manchester City,eng Premier League,26.0,1994.0,13,11,953,...,95.9,4.72,1.13,0.28,0.66,0.75,50.6,46.6,92.2,2.74
614,John Stones,eng ENG,DF,Manchester City,eng Premier League,26.0,1994.0,22,22,1933,...,264.6,9.44,1.53,0.00,0.19,0.05,72.3,70.9,98.1,0.28
675,Kyle Walker,eng ENG,DF,Manchester City,eng Premier League,30.0,1990.0,24,22,1946,...,182.8,8.52,3.06,0.28,0.37,0.97,75.2,72.7,96.6,1.67
691,Oleksandr Zinchenko,ua UKR,DF,Manchester City,eng Premier League,23.0,1996.0,20,15,1478,...,151.3,7.80,2.32,0.12,0.49,0.30,76.0,72.4,95.3,2.07
870,Gabriel Jesus,br BRA,FW,Manchester City,eng Premier League,23.0,1997.0,29,22,2063,...,85.6,5.15,1.22,1.09,1.97,2.18,53.1,35.2,66.1,10.90
905,Riyad Mahrez,dz ALG,FW,Manchester City,eng Premier League,29.0,1991.0,27,23,1949,...,139.8,9.31,2.07,2.21,1.52,1.61,59.5,49.6,83.3,8.02


## Goleiros

**Saída de bola**
- `PassCmp%`: Precisão geral de passe — mede se o goleiro consegue sair jogando sob pressão
- `LongCmp%`: Precisão em lançamentos longos — fundamental para o estilo de Guardiola que explora o espaço atrás da linha

**Posicionamento e participação**
- `Touches/90`: Volume de toques — goleiros ativos têm valores elevados por atuarem como 11º jogador de linha
- `%TchsDefPen`: % de toques dentro da própria área — valores baixos indicam posicionamento avançado (sweeper-keeper)
- `ProgPass/90`: Passes progressivos — quantifica a contribuição direta na construção ofensiva

> **Sugestão:** Comparar Ederson com os demais goleiros do dataset em `ProgPass/90` e `LongCmp%` revela o quanto seu perfil de sweeper-keeper é diferenciado. Adicionar `PrgDistPass/90` quantifica melhor a extensão dessa contribuição.

In [119]:
gk = city_players[city_players["Pos"] == "GK"].copy()
metrics_gk = ["Player", "PassCmp%", "LongCmp%", "Touches/90", "ProgPass/90", "%TchsDefPen"]
display(gk[metrics_gk])

,Player,PassCmp%,LongCmp%,Touches/90,ProgPass/90,%TchsDefPen
1254,Ederson,83.1,59.9,31.9,0.03,0.874608


## Defensores

### Métricas defensivas

**Volume e eficiência de desarmes**
- `TklAtt/90`: Tentativas de desarme — indica agressividade defensiva
- `TklW/90`: Desarmes bem-sucedidos — quantifica a efetividade real
- `Tkl%vDrib`: Taxa de sucesso contra dribles — mede qualidade em duelos 1v1

**Distribuição espacial dos desarmes**
- `Def 3rdTkl/90`: Desarmes no terço defensivo — indica jogo reativo / bloco baixo
- `Mid 3rdTkl/90`: Desarmes no meio-campo — indica cobertura territorial mais ampla
- `Att 3rdTkl/90`: Desarmes no terço ofensivo — indica pressing e linha defensiva alta

**Antecipação e reatividade**
- `Interceptions/90`: Interceptações — mede leitura de jogo e antecipação de jogadas
- `Clearances/90`: Cortes — alívio sob pressão, típico de defesas de bloco baixo
- `Blocks/90`, `ShotBlocks/90`: Bloqueios totais e de finalização — ação reativa direta

### Índices calculados

**Eficiência de desarme**
$$Tkl\_Eff = \frac{TklW/90}{TklAtt/90}$$
Proporção das tentativas de desarme com sucesso.

**Eficiência contra dribles**
$$Dribble\_Eff = \frac{TklvDribW/90}{TklvDribAtt/90}$$
Capacidade em situações de 1 contra 1.

**Índice de antecipação**
$$Anticipation = \frac{Interceptions}{Interceptions + Clearances}$$
Alta antecipação = leitura de jogo; alta reatividade = cortes reativos.

**Índice de reatividade**
$$Reactivity = Blocks + Clearances$$

Normalização Min-Max (0→1) calculada em relação a **todos os defensores das 5 ligas**, permitindo comparar os jogadores do City dentro da distribuição real da posição.

### Classificação de perfil
- **Proativo (linha alta):** > 40% dos desarmes no meio-campo
- **Reativo (bloco baixo):** > 60% dos desarmes no terço defensivo
- **Equilibrado:** distribuição intermediária

> **Sugestão:** Incluir `ProgPassReceived/90` para medir o quanto os defensores recebem bolas progressivas — indicativo do papel deles na fase de construção ofensiva do City.

In [120]:
def process_defensive_metrics(data, squad="Manchester City"):
    # Use all defenders from the full dataset so normalization reflects the position distribution
    df_all = data[data["Pos"] == "DF"].copy()

    # Derived metrics
    df_all["Tkl_Eff"] = df_all["TklW/90"] / df_all["TklAtt/90"].replace(0, float("nan"))

    total_tkl = df_all["Def 3rdTkl/90"] + df_all["Mid 3rdTkl/90"] + df_all["Att 3rdTkl/90"]
    df_all["Def_3rd_%"] = df_all["Def 3rdTkl/90"] / total_tkl.replace(0, float("nan"))
    df_all["Mid_3rd_%"] = df_all["Mid 3rdTkl/90"] / total_tkl.replace(0, float("nan"))
    df_all["Att_3rd_%"] = df_all["Att 3rdTkl/90"] / total_tkl.replace(0, float("nan"))

    df_all["Dribble_Eff"] = df_all["TklvDribW/90"] / df_all["TklvDribAtt/90"].replace(0, float("nan"))
    df_all["Anticipation"] = df_all["Interceptions/90"] / (
        df_all["Interceptions/90"] + df_all["Clearances/90"]).replace(0, float("nan"))
    df_all["Reactivity"] = df_all["Blocks/90"] + df_all["Clearances/90"]
    df_all["Progression"] = df_all["ProgPass/90"] + df_all["Carries/90"]
    df_all["Involvement"] = df_all["PassCmp/90"]

    # Normalize using the full defender pool as reference (not just City)
    norm_cols = ["Tkl_Eff", "Dribble_Eff", "Anticipation", "Reactivity", "Mid_3rd_%"]
    for col in norm_cols:
        mn, mx = df_all[col].min(), df_all[col].max()
        df_all[f"{col}_norm"] = (df_all[col] - mn) / (mx - mn) if mx > mn else 0

    # Profile classification based on territorial distribution
    def classify_profile(row):
        if row["Mid_3rd_%"] > 0.4:
            return "Proativo (Linha Alta)"
        elif row["Def_3rd_%"] > 0.6:
            return "Reativo (Bloco Baixo)"
        return "Equilibrado"

    df_all["Perfil_Estilo"] = df_all.apply(classify_profile, axis=1)

    # Flags: top 25% threshold computed across all defenders, not just City
    df_all["Elite_1v1"] = df_all["Dribble_Eff"] > df_all["Dribble_Eff"].quantile(0.75)
    df_all["High_Anticipation"] = df_all["Anticipation"] > df_all["Anticipation"].quantile(0.75)
    df_all["Efficient_Defender"] = df_all["Tkl_Eff"] > df_all["Tkl_Eff"].quantile(0.75)

    # Filter to target squad only for display
    df_city = df_all[df_all["Squad"] == squad].copy()

    cols_def = ["Player", "Tkl_Eff", "Dribble_Eff", "Anticipation", "Reactivity",
                "Def_3rd_%", "Mid_3rd_%", "Att_3rd_%", "Perfil_Estilo",
                "Elite_1v1", "High_Anticipation", "Efficient_Defender"]
    cols_off = ["Player", "PassCmp/90", "ProgPass/90", "Carries/90", "Progression", "Involvement"]

    df_def = df_city[cols_def]
    df_off = df_city[cols_off]

    print("Características Defensivas")
    display(df_def)
    print("Características Ofensivas")
    display(df_off)

    return df_def, df_off


# Pass full dataset so normalization uses all defenders across the 5 leagues
df_defensive, df_offensive = process_defensive_metrics(data)

Características Defensivas


,Player,Tkl_Eff,Dribble_Eff,Anticipation,Reactivity,Def_3rd_%,Mid_3rd_%,Att_3rd_%,Perfil_Estilo,Elite_1v1,High_Anticipation,Efficient_Defender
15,Nathan Aké,0.396450,0.803571,0.186630,3.59,0.470238,0.398810,0.130952,Equilibrado,True,False,False
113,João Cancelo,0.572848,0.461318,0.591304,2.90,0.533113,0.337748,0.129139,Equilibrado,False,True,False
189,Rúben Dias,0.673469,0.260274,0.156600,5.19,0.551020,0.387755,0.061224,Equilibrado,False,False,True
366,Aymeric Laporte,0.459770,0.333333,0.131707,4.77,0.770115,0.229885,0.000000,Reativo (Bloco Baixo),False,False,False
445,Benjamin Mendy,0.661972,0.276471,0.354037,3.30,0.198582,0.468085,0.333333,Proativo (Linha Alta),False,True,True
614,John Stones,0.602151,0.507692,0.206208,5.02,0.652174,0.250000,0.097826,Reativo (Bloco Baixo),False,False,False
675,Kyle Walker,0.730994,0.543210,0.281633,3.10,0.511628,0.377907,0.110465,Equilibrado,True,False,True
691,Oleksandr Zinchenko,0.617188,0.497076,0.289786,4.58,0.429688,0.335938,0.234375,Equilibrado,False,False,False


Características Ofensivas


,Player,PassCmp/90,ProgPass/90,Carries/90,Progression,Involvement
15,Nathan Aké,78.9,4.04,68.2,72.24,78.9
113,João Cancelo,70.4,6.51,63.8,70.31,70.4
189,Rúben Dias,79.1,3.67,68.1,71.77,79.1
366,Aymeric Laporte,77.9,4.83,68.0,72.83,77.9
445,Benjamin Mendy,50.2,4.62,41.4,46.02,50.2
614,John Stones,75.1,3.49,65.7,69.19,75.1
675,Kyle Walker,78.5,4.95,69.5,74.45,78.5
691,Oleksandr Zinchenko,81.2,5.67,61.4,67.07,81.2


In [121]:
def plot_territorial_distribution(df):
    # Sort by attacking third to highlight proactive defenders
    df_sorted = df.sort_values("Att_3rd_%")

    fig = go.Figure()
    fig.add_trace(go.Bar(y=df_sorted["Player"], x=df_sorted["Def_3rd_%"],
                         name="Terço Defensivo", orientation="h", marker_color="#e74c3c"))
    fig.add_trace(go.Bar(y=df_sorted["Player"], x=df_sorted["Mid_3rd_%"],
                         name="Meio-Campo", orientation="h", marker_color="#f1c40f"))
    fig.add_trace(go.Bar(y=df_sorted["Player"], x=df_sorted["Att_3rd_%"],
                         name="Terço Ofensivo", orientation="h", marker_color="#2ecc71"))

    fig.update_layout(
        barmode="stack",
        title="Distribuição Territorial de Ações Defensivas",
        xaxis_title="Proporção de Desarmes",
        xaxis_tickformat=".0%",
        template="plotly_white"
    )
    fig.show()

plot_territorial_distribution(df_defensive)

In [122]:
def plot_defensive_style(df):
    # Bubble size = tackle efficiency; color = dribble defense quality
    fig = px.scatter(df, x="Anticipation", y="Reactivity",
                     size="Tkl_Eff", color="Dribble_Eff",
                     hover_name="Player",
                     title="Estilo Defensivo: Antecipação vs. Reatividade",
                     labels={"Anticipation": "Índice de Antecipação",
                             "Reactivity": "Índice de Reatividade (Bloqueios + Cortes)"},
                     template="plotly_white")

    fig.add_hline(y=df["Reactivity"].mean(), line_dash="dot", annotation_text="Média")
    fig.add_vline(x=df["Anticipation"].mean(), line_dash="dot", annotation_text="Média")
    fig.show()

plot_defensive_style(df_defensive)

In [123]:
def plot_offensive_buildup(df):
    # Bubble size = progressive passes; color = carries volume
    fig = px.scatter(df, x="Involvement", y="Progression",
                     size="ProgPass/90", color="Carries/90",
                     hover_name="Player",
                     title="Construção: Envolvimento vs. Progressão",
                     labels={"Involvement": "Envolvimento (Passes Completos/90)",
                             "Progression": "Progressão (Passes Prog + Conduções)",
                             "Carries/90": "Conduções/90"},
                     color_continuous_scale=px.colors.sequential.Viridis,
                     template="plotly_white")

    fig.add_hline(y=df["Progression"].mean(), line_dash="dash", opacity=0.5)
    fig.add_vline(x=df["Involvement"].mean(), line_dash="dash", opacity=0.5)
    fig.show()

plot_offensive_buildup(df_offensive)

## Meio-Campistas

### Métricas extraídas

**Criatividade e quebra de linhas**
- `KeyPass/90`: Passes que levam diretamente a uma finalização
- `ProgPass/90`: Passes que movem a bola significativamente em direção ao gol adversário
- `PassIntoBox/90`: Passes completados dentro da grande área (excluindo cruzamentos)
- `TBCmp/90`: Through balls completados — passes que rompem a última linha defensiva
- `SCA/90`: Shot-creating actions — ações que resultam diretamente em chute

**Condução e progressão**
- `PrgDistCarry/90`: Distância de condução progressiva — ameaça carregando a bola

**Pressão defensiva (gegenpressing)**
- `PressAtt/90`: Volume de pressões sobre o adversário
- `SuccPress/90`: Pressões que resultam em recuperação da bola
- `Att 3rdPress/90`: Pressões no terço de ataque — gatilho do pressing de Guardiola

**Retenção e segurança de posse**
- `PassCmp%`: Percentual de acerto de passe — base de qualidade na circulação
- `PassUnderPress/90`: Passes executados enquanto sofria pressão — resistência sob marcação
- `Touches/90`: Volume total de participação no jogo

### Índices calculados

**Índice de Verticalidade**
$$\frac{PrgDistPass/90}{TotDistPass/90}$$
Mede quanto do volume de passe é para frente em vez de lateral/para trás.

**Taxa de Sucesso em Pressão**
$$\frac{SuccPress/90}{PressAtt/90}$$
Eficácia das investidas defensivas sem a bola.

**Agressividade de Recuperação no Ataque**
$$Att3rdTkl/90 + Att3rdPress/90$$
Identifica os "perdigueiros" que sufocam a saída de bola adversária.

**Segurança da Posse**
$$PassCmp\% \times \left(1 - \frac{Miscontrol/90 + Dispossessed/90}{Touches/90}\right)$$
Avalia quem melhor protege a bola considerando volume de toques.

**Perigo por Condução**
$$\frac{CarryIntoBox/90 + CarryIntoThird/90}{Carries/90}$$
Percentual de conduções que resultam em ganho territorial significativo.

Todas as normalizações no radar usam **todos os meio-campistas das 5 ligas** como referência.

### Perfis táticos

| Perfil | Métricas-chave | Exemplo |
|---|---|---|
| **Metrônomo** | Alta `Seguranca_Posse`, alto `Touches/90` | Rodri |
| **Criador** | Alto `KeyPass/90`, `PassIntoBox/90`, `SCA/90` | De Bruyne |
| **Transportador** | Alto `PrgDistCarry/90`, `Perigo_Conducao` | Bernardo Silva |
| **Pressionador** | Alta `Agressividade_Ataque`, `Taxa_Sucesso_Pressao` | Phil Foden |

> **Sugestão:** Incluir `xA/90` e `npxG/90` para medir a qualidade esperada da criação ofensiva — não apenas volume, mas efetividade.

In [124]:
def analyze_midfielders(data, squad="Manchester City"):
    # All MF players across all leagues for position-relative normalization
    df_all = data[data["Pos"].str.contains("MF", na=False)].copy()

    # Derived metrics computed on the full position pool
    df_all["Indice_Verticalidade"] = np.where(
        df_all["TotDistPass/90"] > 0, df_all["PrgDistPass/90"] / df_all["TotDistPass/90"], 0)

    df_all["Taxa_Sucesso_Pressao"] = np.where(
        df_all["PressAtt/90"] > 0, df_all["SuccPress/90"] / df_all["PressAtt/90"], 0)

    df_all["Agressividade_Ataque"] = df_all["Att 3rdTkl/90"] + df_all["Att 3rdPress/90"]

    # Penalizes high pass accuracy if the player frequently loses the ball
    df_all["Seguranca_Posse"] = np.where(
        df_all["Touches/90"] > 0,
        df_all["PassCmp%"] * (1 - (df_all["Miscontrol/90"] + df_all["Dispossessed/90"]) / df_all["Touches/90"]),
        0)

    df_all["Perigo_Conducao"] = np.where(
        df_all["Carries/90"] > 0,
        (df_all["CarryIntoBox/90"] + df_all["CarryIntoThird/90"]) / df_all["Carries/90"],
        0)

    # Filter to target squad for display
    df_city = df_all[df_all["Squad"] == squad].copy()

    id_cols = ["Player", "Pos", "Age", "Min", "90s"]

    cols_off = ["KeyPass/90", "ProgPass/90", "PassIntoBox/90", "TBCmp/90",
                "SCA/90", "PrgDistCarry/90", "PassUnderPress/90", "PassCmp%", "Touches/90",
                "Indice_Verticalidade", "Seguranca_Posse", "Perigo_Conducao"]

    cols_def = ["TklW/90", "Def 3rdTkl/90", "Mid 3rdTkl/90", "Att 3rdTkl/90",
                "PressAtt/90", "Def 3rdPress/90", "Mid 3rdPress/90", "Att 3rdPress/90",
                "PassBlk/90", "Interceptions/90", "SuccPress/90",
                "Taxa_Sucesso_Pressao", "Agressividade_Ataque"]

    df_off = df_city[id_cols + cols_off].sort_values("Min", ascending=False).reset_index(drop=True)
    df_def = df_city[id_cols + cols_def].sort_values("Min", ascending=False).reset_index(drop=True)

    print("Características Defensivas")
    display(df_def)
    print("Características Ofensivas")
    display(df_off)

    # Return df_all so radar normalization can reference the full position pool
    return df_off, df_def, df_all


# Pass full dataset — function filters MF internally
df_off_mf, df_def_mf, df_all_mf = analyze_midfielders(data)

Características Defensivas


,Player,Pos,Age,Min,90s,TklW/90,Def 3rdTkl/90,Mid 3rdTkl/90,Att 3rdTkl/90,PressAtt/90,Def 3rdPress/90,Mid 3rdPress/90,Att 3rdPress/90,PassBlk/90,Interceptions/90,SuccPress/90,Taxa_Sucesso_Pressao,Agressividade_Ataque
0,Rodri,MF,24.0,2748,30.5,1.97,1.05,1.25,0.43,16.90,5.57,8.92,2.39,0.95,1.31,6.46,0.382249,2.82
1,Bernardo Silva,"MF,FW",25.0,2065,22.9,0.92,0.52,0.61,0.26,16.40,3.10,7.60,5.68,0.83,0.92,4.85,0.295732,5.94
2,İlkay Gündoğan,MF,29.0,2029,22.5,0.80,0.40,0.44,0.27,9.56,2.04,4.53,2.98,0.49,0.84,2.89,0.302301,3.25
3,Kevin De Bruyne,MF,29.0,1997,22.2,1.40,0.36,0.86,0.45,16.90,2.07,7.88,6.98,1.13,0.72,5.45,0.322485,7.43
4,Phil Foden,"FW,MF",20.0,1616,18.0,0.89,0.56,0.61,0.28,17.10,2.83,6.50,7.72,0.78,0.67,4.89,0.285965,8.00
5,Ferrán Torres,"FW,MF",20.0,1306,14.5,0.55,0.41,0.41,0.21,15.10,2.69,6.41,6.00,1.10,0.48,4.62,0.305960,6.21
6,Fernandinho,MF,35.0,1188,13.2,1.67,0.45,1.29,0.61,16.90,3.56,10.80,2.58,1.06,1.97,5.61,0.331953,3.19
7,Sergio Agüero,"FW,MF",32.0,559,6.2,0.00,0.00,0.16,0.32,14.50,1.29,5.00,8.23,0.65,0.65,4.03,0.277931,8.55


Características Ofensivas


,Player,Pos,Age,Min,90s,KeyPass/90,ProgPass/90,PassIntoBox/90,TBCmp/90,SCA/90,PrgDistCarry/90,PassUnderPress/90,PassCmp%,Touches/90,Indice_Verticalidade,Seguranca_Posse,Perigo_Conducao
0,Rodri,MF,24.0,2748,30.5,0.89,5.11,0.56,0.10,2.39,159.4,13.20,91.5,99.1,0.205253,90.290464,0.026562
1,Bernardo Silva,"MF,FW",25.0,2065,22.9,1.40,3.14,1.40,0.04,3.49,218.5,8.43,89.7,70.7,0.155906,86.769208,0.101268
2,İlkay Gündoğan,MF,29.0,2029,22.5,1.87,5.24,1.56,0.18,3.69,163.5,8.49,91.0,84.8,0.187618,89.336675,0.058553
3,Kevin De Bruyne,MF,29.0,1997,22.2,3.56,7.79,3.56,0.95,6.40,189.6,7.21,76.0,75.8,0.292880,72.881794,0.095140
4,Phil Foden,"FW,MF",20.0,1616,18.0,1.89,3.33,1.33,0.00,3.72,124.3,7.33,82.0,59.4,0.212158,76.781818,0.088813
5,Ferrán Torres,"FW,MF",20.0,1306,14.5,1.24,2.34,1.03,0.14,2.76,83.3,4.83,81.4,42.8,0.180476,75.104813,0.066460
6,Fernandinho,MF,35.0,1188,13.2,0.91,7.35,1.29,0.15,2.20,184.5,9.24,87.8,85.4,0.271866,87.028923,0.059235
7,Sergio Agüero,"FW,MF",32.0,559,6.2,0.97,1.94,0.97,0.00,2.58,52.9,4.68,84.1,39.7,0.152773,75.541713,0.053903


In [125]:
df_mf = pd.merge(df_off_mf, df_def_mf, on=['Player', 'Pos', 'Age', 'Min', '90s'])

### Progressão: Passes vs. Condução

Identifica como cada jogador avança a bola — pelo passe ou pela condução. Jogadores no quadrante superior direito são eficazes nas duas dimensões.

In [126]:
def plot_progression_scatter(df):
    fig = px.scatter(df, x="ProgPass/90", y="PrgDistCarry/90",
                     text="Player",
                     title="Progressão: Passes Progressivos vs. Condução",
                     labels={"ProgPass/90": "Passes Progressivos / 90",
                             "PrgDistCarry/90": "Distância de Condução Progressiva / 90"},
                     template="plotly_white")

    fig.update_traces(textposition="top center", marker=dict(size=10, color="#6CABDD"))
    fig.add_hline(y=df["PrgDistCarry/90"].mean(), line_dash="dot", opacity=0.4)
    fig.add_vline(x=df["ProgPass/90"].mean(), line_dash="dot", opacity=0.4)
    fig.show()

plot_progression_scatter(df_mf)

### Perfil Tático (Radar)

Comparação normalizada entre jogadores em cinco dimensões táticas. Valores normalizados em relação a todos os meio-campistas das 5 ligas.

In [127]:
def plot_tactical_radar(df_city, df_all, players):
    metrics = ["Seguranca_Posse", "Indice_Verticalidade", "SCA/90", "Perigo_Conducao", "Agressividade_Ataque"]
    labels = ["Segurança de Posse", "Verticalidade", "SCA/90", "Perigo Condução", "Agressividade"]

    df_r = df_city[df_city["Player"].isin(players)].copy().reset_index(drop=True)

    # Normalize each metric using the min/max of the full position pool
    for col in metrics:
        mn, mx = df_all[col].min(), df_all[col].max()
        df_r[col] = (df_r[col] - mn) / (mx - mn) if mx > mn else 0

    colors = ["#6CABDD", "#1C2C5B", "#FFC65C"]
    fig = go.Figure()

    for i, row in df_r.iterrows():
        values = [row[m] for m in metrics] + [row[metrics[0]]]
        fig.add_trace(go.Scatterpolar(
            r=values,
            theta=labels + [labels[0]],
            fill="toself",
            name=row["Player"],
            line_color=colors[i % len(colors)],
            opacity=0.7
        ))

    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
        title="Perfil Tático Normalizado (referência: todos os MF das 5 ligas)",
        template="plotly_white"
    )
    fig.show()

# df_all_mf provides the normalization reference across the full position pool
plot_tactical_radar(df_mf, df_all_mf, ["Rodri", "Kevin De Bruyne"])

### Mapa de Pressão por Terço

Distribuição das pressões defensivas por terço do campo — revela a postura defensiva e o raio de ação de cada jogador.

In [128]:
def plot_pressure_map(df):
    press_cols = ["Def 3rdPress/90", "Mid 3rdPress/90", "Att 3rdPress/90"]
    df_p = df[["Player"] + press_cols].copy()
    df_p["Total"] = df_p[press_cols].sum(axis=1)
    df_p = df_p.sort_values("Total", ascending=True)

    colors = ["#A8D0E6", "#F8E9A1", "#F76C6C"]
    labels = ["Terço Defensivo", "Terço Médio", "Terço de Ataque"]

    fig = go.Figure()
    for col, color, label in zip(press_cols, colors, labels):
        fig.add_trace(go.Bar(y=df_p["Player"], x=df_p[col],
                             name=label, orientation="h", marker_color=color))

    fig.update_layout(
        barmode="stack",
        title="Mapa de Pressão por Terço do Campo",
        xaxis_title="Pressões / 90",
        template="plotly_white"
    )
    fig.show()

plot_pressure_map(df_mf)

### Resistência à Pressão vs. Segurança de Posse

Quem recebe mais pressão e ainda assim mantém a bola com eficiência — o "porto seguro" do time.

In [129]:
def plot_pressure_resistance(df):
    # Bubble size = total touches; reveals who carries the most load under pressure
    fig = px.scatter(df, x="PassUnderPress/90", y="Seguranca_Posse",
                     size="Touches/90", text="Player",
                     title="Resistência à Pressão vs. Segurança de Posse",
                     labels={"PassUnderPress/90": "Passes Sob Pressão / 90",
                             "Seguranca_Posse": "Segurança de Posse (índice)",
                             "Touches/90": "Toques/90"},
                     template="plotly_white")

    fig.update_traces(textposition="top center")
    fig.add_hline(y=df["Seguranca_Posse"].mean(), line_dash="dot", opacity=0.4)
    fig.add_vline(x=df["PassUnderPress/90"].mean(), line_dash="dot", opacity=0.4)
    fig.show()

plot_pressure_resistance(df_mf)

### Criação de Oportunidades: Passes-Chave e Bolas em Profundidade

Separa o volume de criação (KeyPass) da qualidade/dificuldade do passe (Through Balls).

In [130]:
def plot_chance_creation(df):
    df_c = df[["Player", "KeyPass/90", "TBCmp/90"]].copy()
    df_c["Total"] = df_c["KeyPass/90"] + df_c["TBCmp/90"]
    df_c = df_c.sort_values("Total")

    fig = go.Figure()
    fig.add_trace(go.Bar(y=df_c["Player"], x=df_c["KeyPass/90"],
                         name="Passes-Chave", orientation="h", marker_color="#6CABDD"))
    fig.add_trace(go.Bar(y=df_c["Player"], x=df_c["TBCmp/90"],
                         name="Bolas em Profundidade", orientation="h", marker_color="#1C2C5B"))

    fig.update_layout(
        barmode="stack",
        title="Criação de Oportunidades: Passes-Chave e Bolas em Profundidade",
        xaxis_title="Ações / 90",
        template="plotly_white"
    )
    fig.show()

plot_chance_creation(df_mf)

## Atacantes

### Métricas extraídas

**Finalização e criação**
- `npG/90`: Gols sem pênaltis por 90 — medida pura de produção ofensiva
- `npxG/90`: Expected goals sem pênaltis — qualidade das chances recebidas
- `Shots/90`: Volume de finalizações
- `SoT%`: Percentual de chutes no alvo
- `npxG/Shot`: Qualidade média de cada finalização
- `KeyPass/90`: Passes que geram chutes — atacantes que também criam
- `PassIntoBox/90`: Passes completados dentro da área

**Condução e posicionamento**
- `ProgCarry/90`: Conduções progressivas — entrada de bola no terço final
- `Att PenTchs/90`: Toques na área adversária — presença na zona de finalização
- `ProgPassReceived/90`: Passes progressivos recebidos — mobilidade e movimentação sem bola

**Pressão defensiva (sem a bola)**
- `Att 3rdPress/90`: Pressões no terço de ataque — participação no gegenpressing
- `SuccPress/90`: Pressões com recuperação efetiva de posse
- `Att 3rdTkl/90`: Desarmes no terço ofensivo — agressividade defensiva alta

### Índices calculados

**Clinical Finishing Index**
$$npG/90 - npxG/90$$
Capacidade de superar o xG. Positivo = finalizador clínico; negativo = desperdiçando chances.

**Índice de Verticalidade**
$$\frac{ProgPass/90 + PassIntoBox/90}{PassCmp/90}$$
Proporção de passes que quebram linhas em relação ao total completado.

**Eficiência de Assistência**
$$Ast/90 - xA/90$$
Diferença entre assistências reais e esperadas — conversão de bolas-passe em gols.

**Box Efficiency**
$$\frac{Att\ PenTchs/90}{Shots/90}$$
Toques na área por finalização — indica precisão de posicionamento.

**Retention Progression**
$$(ProgCarry/90 + ProgPassReceived/90) - (Miscontrol/90 + Dispossessed/90)$$
Ganhos de território líquidos — progressão real descontando perdas de bola.

**High Recovery Rate**
$$Att3rdTkl/90 + Att3rdPress/90$$
Ações defensivas no terço final — atacantes que pressionam com efetividade.

**Press Efficiency %**
$$\frac{SuccPress/90}{PressAtt/90} \times 100$$
Qualidade do pressing — não apenas volume, mas efetividade da pressão.

Todas as normalizações no radar usam **todos os atacantes das 5 ligas** como referência.


In [136]:
def analyze_forwards(data, squad="Manchester City"):
    # All FW players across all leagues for position-relative normalization
    fw_positions = ["FW", "DF,FW", "FW,DF", "FW,MF", "MF,FW"]
    df_all = data[data["Pos"].isin(fw_positions)].copy()

    # Derived metrics computed on the full position pool
    df_all["Verticality_Index"] = (
        df_all["ProgPass/90"] + df_all["PassIntoBox/90"]
    ) / df_all["PassCmp/90"].replace(0, float("nan"))

    df_all["Assists_Efficiency"] = df_all["Ast/90"] - df_all["xA/90"]

    # Positive = clinical finisher; negative = missing chances relative to xG
    df_all["Clinical_Finishing_Index"] = df_all["npG/90"] - df_all["npxG/90"]

    df_all["Box_Efficiency"] = df_all["Att PenTchs/90"] / df_all["Shots/90"].replace(0, float("nan"))

    df_all["Retention_Progression"] = (
        df_all["ProgCarry/90"] + df_all["ProgPassReceived/90"]
    ) - (df_all["Miscontrol/90"] + df_all["Dispossessed/90"])

    df_all["High_Recovery_Rate"] = df_all["Att 3rdTkl/90"] + df_all["Att 3rdPress/90"]

    df_all["Press_Efficiency_Pct"] = (
        df_all["SuccPress/90"] / df_all["PressAtt/90"].replace(0, float("nan"))
    ) * 100

    # Filter to target squad for display
    df_city = df_all[df_all["Squad"] == squad].copy()

    id_cols = ["Player", "Nation", "Pos", "Age", "90s"]

    cols_off = id_cols + [
        "npG/90", "npxG/90", "Clinical_Finishing_Index", "Shots/90", "SoT%",
        "npxG/Shot", "KeyPass/90", "PassIntoBox/90", "Verticality_Index",
        "Assists_Efficiency", "ProgCarry/90", "Retention_Progression", "Att PenTchs/90", "ProgPass/90"
    ]
    cols_def = id_cols + [
        "Att 3rdPress/90", "SuccPress/90", "Press_Efficiency_Pct",
        "Att 3rdTkl/90", "High_Recovery_Rate", "Interceptions/90", "Blocks/90"
    ]

    df_off = df_city[cols_off].copy()
    df_def = df_city[cols_def].copy()

    print("Características Ofensivas")
    display(df_off)
    print("Características Defensivas")
    display(df_def)

    # Return df_all so radar normalization can reference the full position pool
    return df_off, df_def, df_all


# Pass full dataset — function filters FW positions internally
df_off_fw, df_def_fw, df_all_fw = analyze_forwards(data)

Características Ofensivas


,Player,Nation,Pos,Age,90s,npG/90,npxG/90,Clinical_Finishing_Index,Shots/90,SoT%,npxG/Shot,KeyPass/90,PassIntoBox/90,Verticality_Index,Assists_Efficiency,ProgCarry/90,Retention_Progression,Att PenTchs/90,ProgPass/90
870,Gabriel Jesus,br BRA,FW,23.0,22.9,0.39,0.38,0.01,2.36,29.6,0.16,1.22,0.61,0.070251,0.02,5.15,11.90,6.59,1.35
905,Riyad Mahrez,dz ALG,FW,29.0,21.7,0.42,0.26,0.16,2.67,37.9,0.10,1.89,2.40,0.153444,0.01,9.31,14.20,7.19,4.06
984,Raheem Sterling,eng ENG,FW,25.0,28.2,0.35,0.41,-0.06,2.38,41.8,0.17,1.35,1.03,0.094721,0.08,8.30,12.87,8.51,2.20
1024,Sergio Agüero,ar ARG,"FW,MF",32.0,6.2,0.48,0.29,0.19,2.74,58.8,0.11,0.97,0.97,0.125974,-0.05,3.87,8.38,8.06,1.94
1068,Phil Foden,eng ENG,"FW,MF",20.0,18.0,0.50,0.31,0.19,2.50,44.4,0.12,1.89,1.33,0.121990,0.05,7.61,12.66,8.00,3.33
1208,Ferrán Torres,es ESP,"FW,MF",20.0,14.5,0.48,0.37,0.11,2.48,38.9,0.15,1.24,1.03,0.135887,0.01,5.03,7.72,4.83,2.34
1937,Bernardo Silva,pt POR,"MF,FW",25.0,22.9,0.09,0.12,-0.03,1.22,46.4,0.10,1.40,1.40,0.081216,0.10,12.10,14.72,4.41,3.14


Características Defensivas


,Player,Nation,Pos,Age,90s,Att 3rdPress/90,SuccPress/90,Press_Efficiency_Pct,Att 3rdTkl/90,High_Recovery_Rate,Interceptions/90,Blocks/90
870,Gabriel Jesus,br BRA,FW,23.0,22.9,7.60,4.50,31.034483,0.39,7.99,0.83,0.83
905,Riyad Mahrez,dz ALG,FW,29.0,21.7,5.25,2.72,23.247863,0.37,5.62,0.28,1.11
984,Raheem Sterling,eng ENG,FW,25.0,28.2,4.08,2.94,28.823529,0.11,4.19,0.50,0.60
1024,Sergio Agüero,ar ARG,"FW,MF",32.0,6.2,8.23,4.03,27.793103,0.32,8.55,0.65,0.81
1068,Phil Foden,eng ENG,"FW,MF",20.0,18.0,7.72,4.89,28.596491,0.28,8.00,0.67,0.83
1208,Ferrán Torres,es ESP,"FW,MF",20.0,14.5,6.00,4.62,30.596026,0.21,6.21,0.48,1.17
1937,Bernardo Silva,pt POR,"MF,FW",25.0,22.9,5.68,4.85,29.573171,0.26,5.94,0.92,0.87


### Eficiência de Finalização

Identifica finalizadores clínicos (convertem acima do xG) vs. jogadores que desperdiçam chances.
Quadrante superior direito: alta qualidade de chance **e** alta eficiência clínica.

In [137]:
def plot_finishing_efficiency(df_off):
    # Bubble size = shot volume; y=0 is the "expectation line"
    fig = px.scatter(df_off, x="npxG/90", y="Clinical_Finishing_Index",
                     size="Shots/90", text="Player",
                     title="Qualidade de Chance vs. Eficiência Clínica",
                     labels={"npxG/90": "Qualidade das Chances Recebidas (npxG/90)",
                             "Clinical_Finishing_Index": "Performance acima do xG"},
                     template="plotly_white")

    fig.update_traces(textposition="top center")
    fig.add_hline(y=0, line_dash="dash", line_color="grey", opacity=0.6,
                  annotation_text="Linha do xG")
    fig.add_vline(x=df_off["npxG/90"].mean(), line_dash="dot", line_color="grey", opacity=0.6)
    fig.show()

plot_finishing_efficiency(df_off_fw)

### Perfil Tático (Radar)

Comparação normalizada entre dois atacantes em seis dimensões. Valores normalizados em relação a todos os atacantes das 5 ligas.

In [138]:
def plot_forward_radar(df_off, df_def, df_all, players):
    # Merge to get both offensive and defensive computed metrics in one row
    df_city = df_off.merge(
        df_def[["Player", "High_Recovery_Rate", "Press_Efficiency_Pct"]], on="Player")

    metrics = ["Verticality_Index", "Clinical_Finishing_Index", "Retention_Progression",
               "High_Recovery_Rate", "KeyPass/90", "Att PenTchs/90"]
    labels = ["Verticalidade", "Efic. Finalização", "Retenção Prog.",
              "Recuperação Alta", "Passes-Chave", "Toques na Área"]

    df_r = df_city[df_city["Player"].isin(players)].copy().reset_index(drop=True)

    # Normalize using the min/max of the full forward pool
    for col in metrics:
        mn, mx = df_all[col].min(), df_all[col].max()
        df_r[col] = (df_r[col] - mn) / (mx - mn) if mx > mn else 0

    colors = ["#6CABDD", "#1C2C5B", "#FFC65C"]
    fig = go.Figure()

    for i, row in df_r.iterrows():
        values = [row[m] for m in metrics] + [row[metrics[0]]]
        fig.add_trace(go.Scatterpolar(
            r=values,
            theta=labels + [labels[0]],
            fill="toself",
            name=row["Player"],
            line_color=colors[i % len(colors)],
            opacity=0.7
        ))

    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
        title="Perfil Tático Normalizado (referência: todos os FW das 5 ligas)",
        template="plotly_white"
    )
    fig.show()

# df_all_fw provides the normalization reference across the full position pool
plot_forward_radar(df_off_fw, df_def_fw, df_all_fw, ["Gabriel Jesus", "Raheem Sterling"])

### Sistema de Pressão Alta

Foco no desempenho sem a bola. Quem é o "perdigueiro" mais eficiente do ataque no pressing de Guardiola?

In [139]:
def plot_pressing_performance(df_def):
    # Bubble size and color = combined recovery rate (tackles + pressures in att. third)
    fig = px.scatter(df_def, x="Att 3rdPress/90", y="Press_Efficiency_Pct",
                     size="High_Recovery_Rate", color="High_Recovery_Rate",
                     text="Player",
                     title="Atacantes: Intensidade vs. Eficiência de Pressão Alta",
                     labels={"Att 3rdPress/90": "Pressões no Terço de Ataque / 90",
                             "Press_Efficiency_Pct": "Taxa de Sucesso na Pressão (%)",
                             "High_Recovery_Rate": "Recuperação Alta"},
                     color_continuous_scale="Viridis",
                     template="plotly_white")

    fig.update_traces(textposition="top center")
    fig.add_hline(y=df_def["Press_Efficiency_Pct"].mean(), line_dash="dot", opacity=0.4)
    fig.add_vline(x=df_def["Att 3rdPress/90"].mean(), line_dash="dot", opacity=0.4)
    fig.show()

plot_pressing_performance(df_def_fw)

### Estilo de Progressão

Como cada atacante faz o time ganhar campo — pelo passe progressivo ou pela condução.

In [140]:
def plot_progression_style(df_off):
    df_p = df_off[["Player", "ProgPass/90", "ProgCarry/90"]].copy()
    df_p["Total"] = df_p["ProgPass/90"] + df_p["ProgCarry/90"]
    df_p = df_p.sort_values("Total", ascending=True)

    fig = go.Figure()
    fig.add_trace(go.Bar(y=df_p["Player"], x=df_p["ProgPass/90"],
                         name="Passes Progressivos", orientation="h", marker_color="#6CABDD"))
    fig.add_trace(go.Bar(y=df_p["Player"], x=df_p["ProgCarry/90"],
                         name="Conduções Progressivas", orientation="h", marker_color="#1C2C5B"))

    fig.update_layout(
        barmode="stack",
        title="Estilo de Progressão dos Atacantes",
        xaxis_title="Ações Progressivas / 90",
        template="plotly_white"
    )
    fig.show()

plot_progression_style(df_off_fw)